# 07 CRM Priority Scoring

This notebook combines lifecycle segmentation, lapse-risk ML, and second-purchase propensity into a single CRM action-priority layer.

Inputs:

- `../data/processed/customer_segments.csv`
- `../data/processed/lapse_risk_scores.csv`
- `../data/processed/second_purchase_propensity_scores.csv`

Outputs:

- `../data/processed/crm_customer_priority_scores.csv`
- `../data/processed/crm_priority_summary.csv`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PROCESSED_DIR = Path("../data/processed")
SEGMENTS_PATH = PROCESSED_DIR / "customer_segments.csv"
LAPSE_PATH = PROCESSED_DIR / "lapse_risk_scores.csv"
PROPENSITY_PATH = PROCESSED_DIR / "second_purchase_propensity_scores.csv"

for path in [SEGMENTS_PATH, LAPSE_PATH, PROPENSITY_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required input not found: {path}")

## Load Inputs

The customer segment table is the base layer. Lapse-risk and second-purchase propensity scores are joined where available.

In [ ]:
segments = pd.read_csv(SEGMENTS_PATH, parse_dates=["first_purchase", "last_purchase"])
lapse_scores = pd.read_csv(LAPSE_PATH, parse_dates=["snapshot_date"])
propensity_scores = pd.read_csv(PROPENSITY_PATH, parse_dates=["order_date"])

latest_lapse_snapshot = lapse_scores["snapshot_date"].max()
latest_lapse_scores = lapse_scores[lapse_scores["snapshot_date"] == latest_lapse_snapshot].copy()

propensity_keep = propensity_scores[[
    "customer_id",
    "second_purchase_propensity_score",
    "propensity_band",
]].copy()

priority = segments.merge(
    latest_lapse_scores[[
        "customer_id",
        "snapshot_date",
        "lapse_risk_score",
        "risk_band",
        "lapsed_next_90_days",
    ]],
    on="customer_id",
    how="left",
).merge(propensity_keep, on="customer_id", how="left")

priority["risk_band"] = priority["risk_band"].fillna("Unscored")
priority["propensity_band"] = priority["propensity_band"].fillna("Unscored")
priority["lapse_risk_score"] = priority["lapse_risk_score"].fillna(0.0)
priority["second_purchase_propensity_score"] = priority["second_purchase_propensity_score"].fillna(0.0)

print(f"Customers: {len(priority):,}")
print(f"Latest lapse snapshot: {latest_lapse_snapshot.date()}")
priority.head()

## Priority Logic

The scoring layer converts model outputs into operational CRM actions. High-value/high-risk customers receive the highest priority, followed by second-purchase accelerator eligibility and broader lapse-risk groups.

In [ ]:
priority["value_band"] = np.where(priority["is_high_value"].astype(bool), "High value", "Standard value")

priority["crm_priority_group"] = np.select(
    [
        priority["is_high_value"].astype(bool) & priority["risk_band"].eq("High risk"),
        priority["is_high_value"].astype(bool) & priority["risk_band"].eq("Medium risk"),
        priority["is_second_purchase_accelerator_eligible"].astype(bool),
        priority["risk_band"].eq("High risk"),
        priority["risk_band"].eq("Medium risk"),
        priority["primary_segment"].str.contains("Lapsed", na=False),
    ],
    [
        "P1 High-Value Lapse Prevention",
        "P2 High-Value Watchlist",
        "P3 Second Purchase Acceleration",
        "P4 Standard Lapse Prevention",
        "P5 Standard Nurture",
        "P6 Winback Pool",
    ],
    default="P7 Standard Lifecycle",
)

priority["final_recommended_action"] = priority["crm_priority_group"].map(
    {
        "P1 High-Value Lapse Prevention": "Personalised retention intervention; avoid blanket discounting",
        "P2 High-Value Watchlist": "Light-touch VIP reminder or product discovery",
        "P3 Second Purchase Acceleration": "Day 21 second purchase accelerator test",
        "P4 Standard Lapse Prevention": "Automated replenishment or reactivation trigger",
        "P5 Standard Nurture": "Low-cost reminder or product education",
        "P6 Winback Pool": "Selective winback test with strict incrementality guardrails",
        "P7 Standard Lifecycle": "Maintain standard lifecycle messaging",
    }
)

priority_group_rank = {
    "P1 High-Value Lapse Prevention": 1,
    "P2 High-Value Watchlist": 2,
    "P3 Second Purchase Acceleration": 3,
    "P4 Standard Lapse Prevention": 4,
    "P5 Standard Nurture": 5,
    "P6 Winback Pool": 6,
    "P7 Standard Lifecycle": 7,
}
priority["priority_group_rank"] = priority["crm_priority_group"].map(priority_group_rank)
priority["priority_score"] = (
    priority["is_high_value"].astype(int) * 100
    + priority["lapse_risk_score"] * 60
    + priority["is_second_purchase_accelerator_eligible"].astype(int) * 30
    + priority["second_purchase_propensity_score"] * 10
    - priority["priority_group_rank"] * 5
)
priority = priority.sort_values(["priority_group_rank", "priority_score"], ascending=[True, False]).reset_index(drop=True)
priority["priority_rank"] = range(1, len(priority) + 1)

priority[["customer_id", "crm_priority_group", "final_recommended_action", "priority_rank"]].head(20)

## Priority Summary

The summary quantifies customer counts, revenue exposure, average lapse risk, and recommended CRM action by priority group.

In [ ]:
priority_summary = (
    priority.groupby(["priority_group_rank", "crm_priority_group", "final_recommended_action"], observed=False)
    .agg(
        customers=("customer_id", "count"),
        revenue=("revenue", "sum"),
        avg_lapse_risk_score=("lapse_risk_score", "mean"),
        high_value_customers=("is_high_value", "sum"),
        accelerator_eligible_customers=("is_second_purchase_accelerator_eligible", "sum"),
    )
    .reset_index()
)
priority_summary["customer_pct"] = priority_summary["customers"] / priority_summary["customers"].sum()
priority_summary["revenue_pct"] = priority_summary["revenue"] / priority_summary["revenue"].sum()
priority_summary = priority_summary.sort_values("priority_group_rank").reset_index(drop=True)
priority_summary

In [ ]:
output_columns = [
    "customer_id",
    "priority_rank",
    "priority_score",
    "crm_priority_group",
    "final_recommended_action",
    "primary_segment",
    "value_band",
    "revenue",
    "total_orders",
    "recency_days",
    "is_high_value",
    "lapse_risk_score",
    "risk_band",
    "second_purchase_propensity_score",
    "propensity_band",
    "is_second_purchase_accelerator_eligible",
    "recommended_crm_action",
]
priority[output_columns].to_csv(PROCESSED_DIR / "crm_customer_priority_scores.csv", index=False)
priority_summary.to_csv(PROCESSED_DIR / "crm_priority_summary.csv", index=False)

print(f"Saved priority scores: {len(priority):,} customers")
print(f"Saved priority summary: {len(priority_summary):,} groups")

In [ ]:
expected_outputs = {
    "crm_customer_priority_scores.csv": {"customer_id", "priority_rank", "crm_priority_group", "final_recommended_action", "lapse_risk_score"},
    "crm_priority_summary.csv": {"crm_priority_group", "customers", "revenue", "avg_lapse_risk_score"},
}
for file_name, expected_columns in expected_outputs.items():
    path = PROCESSED_DIR / file_name
    if not path.exists():
        raise FileNotFoundError(f"Expected output was not created: {path}")
    output = pd.read_csv(path)
    missing_columns = sorted(expected_columns - set(output.columns))
    if missing_columns:
        raise ValueError(f"{file_name} is missing columns: {missing_columns}")
    print(f"Validated {file_name}: {output.shape}")